In [2]:
import pandas as pd
import plotly.graph_objects as go
from google.colab import drive
import os
from plotly.subplots import make_subplots
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
os.chdir('/content/drive/MyDrive/CS215/Final ')
#extreme poverty vs child mortality
epdc = pd.read_csv('extreme-poverty-vs-child-mortality.csv')
#Causes of deaths for children
causeOfDeath_1 = pd.read_csv('IHME-GBD_2023_DATA-4ab99a28-1.csv')

#Because of how much data is missing and strange values in the year column im filtering to values greater than 1950 and dropping the na values for the poverty rates
epdc = epdc[ (epdc['Year'] > 1950) & (epdc['Share of population in poverty ($3 a day, 2021 prices)'].notna())]

'''Some of entries are either referring to parts of a country even though theres already an entry for a country while others are reffering to regions that are composed of countries
  already in the dataframe so I dropped them.'''

epdc['Entity'].unique()
dropped_countries =['Bolivia (urban)','China (rural)','China (urban)','Colombia (urban)','East Asia and Pacific (WB)','Ecuador (urban)',
                     'Ethiopia (rural)','Europe and Central Asia (WB)','Honduras (urban)','Latin America and Caribbean (WB)','Micronesia (country) (urban)',
                    'North America (WB)','Rwanda (rural)','Sao Tome and Principe','South Asia (WB)','Sub-Saharan Africa (WB)',
                     'Suriname (urban)','Trinidad and Tobago','Uruguay (urban)','Western and Central Africa (WB)', 'World (excluding China)', 'World (excluding India)','World']
epdc.set_index('Entity',inplace=True)
epdc.drop(dropped_countries,inplace=True)
epdc.drop('World regions according to OWID',axis=1,inplace=True)
epdc

,Code,Year,"Child mortality rate of children aged under five years, per 100 live births","Share of population in poverty ($3 a day, 2021 prices)",Population (historical)
Entity,,,,,
Albania,ALB,1996,3.238855,2.967841,3245680.0
Albania,ALB,2002,2.431431,4.194722,3134095.0
Albania,ALB,2005,1.995576,2.650218,3076156.0
Albania,ALB,2008,1.575325,0.827308,2992933.0
Albania,ALB,2012,1.121760,1.794590,2910003.0
...,...,...,...,...,...
Zambia,ZMB,2015,6.153084,67.892840,16399092.0
Zambia,ZMB,2022,4.666671,71.656150,20152935.0
Zimbabwe,ZWE,2011,7.883335,35.717000,13595421.0


In [4]:
#Before merging datasets I want to sort and drop some columns for so its less cluttered
causeOfDeath_1.sort_values('location',inplace=True)
causeOfDeath_1.set_index('location',inplace=True)
causeOfDeath_1.drop(['measure','sex','age','metric'],axis=1,inplace=True)
causeOfDeath_1

,cause,year,val,upper,lower
location,,,,,
Afghanistan,Unintentional injuries,1984,0.012560,0.020804,0.007114
Afghanistan,Enteric infections,1999,0.134751,0.208191,0.076982
Afghanistan,Chronic respiratory diseases,1992,0.001661,0.003073,0.000917
Afghanistan,Other infectious diseases,1990,0.282495,0.371577,0.196566
Afghanistan,Maternal and neonatal disorders,1990,0.202710,0.241124,0.157737
...,...,...,...,...,...
Zimbabwe,Skin and subcutaneous diseases,1987,0.000338,0.000544,0.000190
Zimbabwe,Diabetes and kidney diseases,1985,0.000888,0.001284,0.000605
Zimbabwe,Transport injuries,1999,0.007074,0.010584,0.004650


In [5]:
merged_df = pd.merge(epdc, causeOfDeath_1,left_index=True,right_index=True,how='inner')


In [6]:
merged_df['cause'].unique()
'''This map is gonna be used to categorize the causes of deaths.For the visualization because of how much data there is
i'll make 3 subplots based on the category and display the aggregated data of the countries in the region'''

key = {'Digestive diseases':1, 'Self-harm and interpersonal violence':1,
       'HIV/AIDS and sexually transmitted infections':3,
       'Nutritional deficiencies':1,
       'Neglected tropical diseases and malaria':3, 'Transport injuries':2,
       'Maternal and neonatal disorders':2, 'Substance use disorders':1,
       'Musculoskeletal disorders':2, 'Unintentional injuries':1, 'Neoplasms':2,
       'Diabetes and kidney diseases':1, 'Skin and subcutaneous diseases':2,
       'Chronic respiratory diseases':2,
       'Respiratory infections and tuberculosis':2, 'Enteric infections':3,
       'Other non-communicable diseases':3, 'Cardiovascular diseases':1,
       'Neurological disorders':2, 'Other infectious diseases':2}

merged_df['category'] = merged_df['cause'].map(key)

merged_df

,Code,Year,"Child mortality rate of children aged under five years, per 100 live births","Share of population in poverty ($3 a day, 2021 prices)",Population (historical),cause,year,val,upper,lower,category
Entity,,,,,,,,,,,
Albania,ALB,1996,3.238855,2.967841,3245680.0,Digestive diseases,1982,0.010850,0.015231,0.007887,1
Albania,ALB,1996,3.238855,2.967841,3245680.0,Self-harm and interpersonal violence,1988,0.013985,0.021712,0.008339,1
Albania,ALB,1996,3.238855,2.967841,3245680.0,HIV/AIDS and sexually transmitted infections,1987,0.001201,0.002403,0.000483,3
Albania,ALB,1996,3.238855,2.967841,3245680.0,Nutritional deficiencies,1996,0.011504,0.014744,0.009007,1
Albania,ALB,1996,3.238855,2.967841,3245680.0,Neglected tropical diseases and malaria,1981,0.000980,0.002303,0.000140,3
...,...,...,...,...,...,...,...,...,...,...,...
Zimbabwe,ZWE,2019,5.106257,49.219894,15271377.0,Skin and subcutaneous diseases,1987,0.000338,0.000544,0.000190,2
Zimbabwe,ZWE,2019,5.106257,49.219894,15271377.0,Diabetes and kidney diseases,1985,0.000888,0.001284,0.000605,1
Zimbabwe,ZWE,2019,5.106257,49.219894,15271377.0,Transport injuries,1999,0.007074,0.010584,0.004650,2


In [7]:
#Downsize to one a few countries
#graph how their rates change over time.

sample = merged_df[(merged_df.index == 'Canada') | (merged_df.index == 'Mexico') | (merged_df.index == 'United Kingdom')
| (merged_df.index == 'United Kingdom')| (merged_df.index == 'China')| (merged_df.index == 'Russia')| (merged_df.index == 'India')]
sample

,Code,Year,"Child mortality rate of children aged under five years, per 100 live births","Share of population in poverty ($3 a day, 2021 prices)",Population (historical),cause,year,val,upper,lower,category
Entity,,,,,,,,,,,
Canada,CAN,1971,2.094232,2.496907,21895694.0,Enteric infections,2001,0.007158,0.008180,0.006131,3
Canada,CAN,1971,2.094232,2.496907,21895694.0,Nutritional deficiencies,1996,0.000312,0.000357,0.000261,1
Canada,CAN,1971,2.094232,2.496907,21895694.0,Musculoskeletal disorders,1986,0.000261,0.000294,0.000229,2
Canada,CAN,1971,2.094232,2.496907,21895694.0,Diabetes and kidney diseases,1985,0.000762,0.000850,0.000686,1
Canada,CAN,1971,2.094232,2.496907,21895694.0,Enteric infections,1986,0.002042,0.002367,0.001755,3
...,...,...,...,...,...,...,...,...,...,...,...
United Kingdom,GBR,2021,0.443539,0.498311,67668788.0,HIV/AIDS and sexually transmitted infections,1994,0.003516,0.003755,0.003310,3
United Kingdom,GBR,2021,0.443539,0.498311,67668788.0,Transport injuries,1993,0.013975,0.016068,0.011939,2
United Kingdom,GBR,2021,0.443539,0.498311,67668788.0,Skin and subcutaneous diseases,1984,0.000151,0.000174,0.000128,2


In [8]:
#The idea is for each cause, there'll be a subplot showing how the poverty and death rates change with time
cat1 = sample[sample['category'] == 1]
titles = cat1['cause'].unique()

cat1_fig = make_subplots(
    rows=2,
    cols=4,
    subplot_titles=titles,
    vertical_spacing=0.10,
    horizontal_spacing=0.15,
    shared_yaxes=True
)


'''Because of the number of traces I tried to use a for loop to add create and add them to the subplot but was having great difficulty.
As a result I used AI to generate the for loop for me'''

for cause_idx, cause in enumerate(titles, 1):  # Start at 1 for row/col calculation
    row = (cause_idx - 1) // 4 + 1  # 4 columns per row
    col = (cause_idx - 1) % 4 + 1

    # Loop through each country for this cause
    for country in cat1.index.unique():
        # Get data for this country and cause
        data = cat1[(cat1.index == country) &
                   (cat1['cause'] == cause)][['year', 'val', 'cause']]\
                  .sort_values('year')\
                  .drop_duplicates()

        # Add trace to the appropriate subplot
        cat1_fig.add_trace(go.Scatter(
            x=data['year'],
            y=data['val'],
            mode='markers',  # or 'lines+markers'
            name=country,
            showlegend=True if cause_idx == 1 else False  # Show legend only once
        ), row=row, col=col)

cat1_fig.update_yaxes(title_text='Cause of death rate')
cat1_fig.update_layout(height=800,title_text='Category 1 Cause of Death Data',showlegend=True)
cat1_fig.show()

In [9]:
cat2 = sample[sample['category'] == 2]
titles = cat2['cause'].unique()

cat2_fig = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=titles,
    vertical_spacing=0.10,
    horizontal_spacing=0.15
)


'''Because of the number of traces I tried to use a for loop to add create and add them to the subplot but was having great difficulty.
As a result I used AI to generate the for loop for me'''

for cause_idx, cause in enumerate(titles, 1):  # Start at 1 for row/col calculation
    row = (cause_idx - 1) // 3 + 1
    col = (cause_idx - 1) % 3 + 1

    # Loop through each country for this cause
    for country in cat2.index.unique():
        # Get data for this country and cause
        data = cat2[(cat2.index == country) &
                   (cat2['cause'] == cause)][['year', 'val', 'cause']]\
                  .sort_values('year')\
                  .drop_duplicates()

        # Add trace to the appropriate subplot
        cat2_fig.add_trace(go.Scatter(
            x=data['year'],
            y=data['val'],
            mode='markers',
            name=country,
            showlegend=True if cause_idx == 1 else False  # Show legend only once
        ), row=row, col=col)

cat2_fig.update_yaxes(title_text='Cause of death rate')
cat2_fig.update_layout(height=800,title_text='Category 2 Cause of Death Data',showlegend=True)
cat2_fig.show()

In [10]:
cat3 = sample[sample['category'] == 3]
titles = cat3['cause'].unique()

cat3_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=titles,
    vertical_spacing=0.10,
    horizontal_spacing=0.15,
    shared_yaxes=True
)



for cause_idx, cause in enumerate(titles, 1):
    row = (cause_idx - 1) // 2 + 1
    col = (cause_idx - 1) % 2 + 1

    # Loop through each country for this cause
    for country in cat3.index.unique():
        # Get data for this country and cause
        data = cat3[(cat3.index == country) &
                   (cat3['cause'] == cause)][['year', 'val', 'cause']]\
                  .sort_values('year')\
                  .drop_duplicates()

        # Add trace to the appropriate subplot
        cat3_fig.add_trace(go.Scatter(
            x=data['year'],
            y=data['val'],
            mode='markers',
            name=country,
            showlegend=True if cause_idx == 1 else False
        ), row=row, col=col)

cat3_fig.update_yaxes(title_text='Cause of death rate')
cat3_fig.update_layout(height=800,title_text='Category 3 Cause of Death Data',showlegend=True)
cat3_fig.show()

In [11]:
poverty_rates = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=sample.index.unique(),
    vertical_spacing=0.10,
    horizontal_spacing=0.15,
    shared_yaxes=True
    )

countries = sample.index.unique()
for idx, country in enumerate(countries, 1):
    row = (idx - 1) // 3 + 1
    col = (idx - 1) % 3 + 1

    data = epdc[epdc.index == country][['Year','Share of population in poverty ($3 a day, 2021 prices)']]\
          .sort_values('Year')

    poverty_rates.add_trace(go.Scatter(
        x=data['Year'],
        y=data['Share of population in poverty ($3 a day, 2021 prices)'],
        mode='markers',
        name=country,
    ),row=row,col=col)
poverty_rates.update_yaxes(title_text='Extreme poverty rate')

poverty_rates.update_layout(height=800,title_text='Extreme Poverty Rates')
poverty_rates.show()